# TürkResearcher — Colab Index Builder

Bu notebook **Colab T4 GPU**'da 650K Türkçe tez abstract'ından kalite filtreli ~500-600K Chroma indeksini kurar ve Hugging Face Hub'a yükler. Lokal RTX 3050 Ti'da 8-14 saat süren iş burada **~1-2 saatte** tamamlanır.

**Önce: Runtime → Change runtime type → T4 GPU seç.**

Akış:
1. Setup + GPU check
2. Google Drive mount (kesinti durumunda devam edebilmek için)
3. HF auth
4. Config
5. Dataset fetch (umutertugrul/turkish-academic-theses-dataset)
6. Quality filter (≥50 kelime, dedupe by tez_no)
7. Chroma index build (mpnet-base-v2, cosine, title+abstract)
8. Smoke retrieval test
9. HF Hub'a upload (filtered parquet + chroma_db)

## 1. Setup + GPU check

In [ ]:
!nvidia-smi
!pip install -q chromadb sentence-transformers datasets huggingface_hub pyarrow tqdm

## 2. Google Drive mount (önerilir — Colab kesintisinde kaldığın yerden devam)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/turkresearcher'
os.makedirs(WORK_DIR, exist_ok=True)
print(f'Working dir: {WORK_DIR}')

# Drive disk durumu (shell magic ayrı satırda olmalı, f-string içinde değil)
!df -h /content/drive/MyDrive | tail -1

## 3. Hugging Face authentication

HF write token gerek (https://huggingface.co/settings/tokens — 'Write' yetkili token oluştur).

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('HF write token: ')
login(hf_token)

## 4. Config

In [ ]:
HF_INPUT_DATASET = 'umutertugrul/turkish-academic-theses-dataset'
HF_OUTPUT_REPO   = 'hakansabunis/tr-academic-research-agent-index'  # build artifacts dataset
EMBED_MODEL      = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
COLLECTION_NAME  = 'turkish_theses'
MIN_WORDS        = 50
BATCH_SIZE       = 1024  # A100=1024, L4=512, T4=256 önerilir
LIMIT            = None  # set to 1000 for smoke; None = full corpus

PERSIST_DIR        = f'{WORK_DIR}/chroma_db'
FILTERED_PARQUET   = f'{WORK_DIR}/abstracts_filtered.parquet'
FILTER_REPORT_JSON = f'{WORK_DIR}/filter_report.json'

## 5. Fetch dataset (~1.56 GB ilk seferde)

In [ ]:
from datasets import load_dataset
import pandas as pd

# token=hf_token açıkça verilir; whoami() Colab vault'a bakar, geçersiz token
# olursa 401 atar — onu atlıyoruz
print(f'Downloading {HF_INPUT_DATASET} ...')
ds = load_dataset(HF_INPUT_DATASET, split='train', token=hf_token)
df = ds.to_pandas()
print(f'Loaded {len(df):,} rows. Cols: {list(df.columns)}')

## 6. Quality filter (aynı mantık, lokal `02_filter_data.py` ile uyumlu)

In [ ]:
import re, json

WS_RE = re.compile(r'\s+')
def _wc(t):
    return 0 if not t else len(WS_RE.findall(t.strip())) + 1
def _norm(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ''
    return str(v).strip()

initial = len(df)
str_cols = ['title_tr','title_en','author','advisor','location','subject',
            'index','status','degree','language','pdf_url','abstract_tr','abstract_en']
for c in str_cols:
    if c in df.columns:
        df[c] = df[c].apply(_norm)
df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype(int)

drops = {}
mask = df['abstract_tr'].str.len() > 0
drops['empty_abstract'] = int((~mask).sum())
df = df[mask].reset_index(drop=True)

mask = df['abstract_tr'].apply(_wc) >= MIN_WORDS
drops['short_abstract'] = int((~mask).sum())
df = df[mask].reset_index(drop=True)

mask = df['title_tr'].str.len() > 0
drops['empty_title'] = int((~mask).sum())
df = df[mask].reset_index(drop=True)

before = len(df)
df = df.drop_duplicates(subset=['tez_no'], keep='first').reset_index(drop=True)
drops['duplicate_tez_no'] = before - len(df)

print(f'Initial: {initial:,}')
for k, v in drops.items():
    print(f'  -{k}: {v:,}')
print(f'Final:   {len(df):,}')

if LIMIT:
    df = df.head(LIMIT).reset_index(drop=True)
    print(f'\nLIMIT applied: {len(df):,}')

df.to_parquet(FILTERED_PARQUET, index=False)
print(f'\nWrote {FILTERED_PARQUET}')

with open(FILTER_REPORT_JSON, 'w', encoding='utf-8') as fp:
    json.dump({
        'initial_rows': initial,
        'final_rows': len(df),
        'drops': drops,
        'min_words': MIN_WORDS,
        'limit': LIMIT,
    }, fp, ensure_ascii=False, indent=2)

## 7. Chroma index build (T4 GPU)

Resumable: kesintide aynı hücreyi tekrar çalıştırmak yeterli — mevcut kayıt sayısından devam eder.

**Tahmini süre (T4):** 600K abstract için ~1.5-2 saat. Smoke (1K) için ~1 dk.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from tqdm.auto import tqdm
import time

os.makedirs(PERSIST_DIR, exist_ok=True)
client = chromadb.PersistentClient(path=PERSIST_DIR)

print(f'Loading embedder on GPU: {EMBED_MODEL}')
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL,
    device='cuda',
)
coll = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embed_fn,
    metadata={'hnsw:space': 'cosine'},
)

existing = coll.count()
print(f'Collection has {existing:,} existing records')

if existing >= len(df):
    print('Index already covers requested range — nothing to do.')
else:
    work_df = df.iloc[existing:].reset_index(drop=True)
    print(f'To embed: {len(work_df):,} (resuming from row {existing:,})')

    def _doc(r):
        t = (r.get('title_tr') or '').strip()
        a = (r.get('abstract_tr') or '').strip()
        return f'{t}\n\n{a}' if t else a

    def _meta(r):
        def s(k):
            v = r.get(k)
            if v is None or (isinstance(v, float) and pd.isna(v)):
                return ''
            return str(v)
        try:
            year_int = int(r.get('year') or 0)
        except (TypeError, ValueError):
            year_int = 0
        try:
            pages_int = int(r.get('pages') or 0)
        except (TypeError, ValueError):
            pages_int = 0
        return {
            'tez_no':   s('tez_no'),
            'title_tr': s('title_tr'),
            'title_en': s('title_en'),
            'author':   s('author'),
            'advisor':  s('advisor'),
            'location': s('location'),
            'subject':  s('subject'),
            'year':     year_int,
            'pages':    pages_int,
            'degree':   s('degree'),
            'pdf_url':  s('pdf_url'),
            'language': s('language'),
        }

    t0 = time.time()
    for i in tqdm(range(0, len(work_df), BATCH_SIZE), desc='indexing'):
        batch = work_df.iloc[i:i+BATCH_SIZE]
        ids = [str(t) for t in batch['tez_no']]
        docs = [_doc(r) for _, r in batch.iterrows()]
        metas = [_meta(r) for _, r in batch.iterrows()]
        coll.upsert(ids=ids, documents=docs, metadatas=metas)
    print(f'\nDone in {(time.time()-t0)/60:.1f} min. Collection now: {coll.count():,}')

## 8. Smoke retrieval test

In [ ]:
queries = [
    'derin öğrenme ile sel tahmini',
    'Türkçe doğal dil işleme metodları',
    'kalp damar hastalıkları teşhis yöntemleri',
    'yenilenebilir enerji rüzgar türbin',
    'üniversite öğrencilerinin akademik başarısı',
]
for q in queries:
    res = coll.query(query_texts=[q], n_results=3)
    metas = res['metadatas'][0] if res['metadatas'] else []
    dists = res['distances'][0] if res['distances'] else []
    print(f'\nq: {q!r}')
    for m, d in zip(metas, dists):
        title = m.get('title_tr', '')[:90]
        print(f'  [{d:.3f}] {m.get("author","?")} ({m.get("year","?")}) — {title}')

## 9. Hugging Face Hub'a yükle

Üretilen artefaktları (filtered parquet + chroma_db) HF Hub'a dataset repo olarak push eder. Sonra lokal makinede `python scripts/04_pull_index_from_hub.py` ile indirilir.

İlk yüklemede `HF_OUTPUT_REPO` adlı public dataset repo'su yoksa otomatik oluşturulur.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HF_OUTPUT_REPO, repo_type='dataset', exist_ok=True, private=False)
print(f'Target: https://huggingface.co/datasets/{HF_OUTPUT_REPO}')

print('\nUploading filter_report.json ...')
api.upload_file(
    path_or_fileobj=FILTER_REPORT_JSON,
    path_in_repo='filter_report.json',
    repo_id=HF_OUTPUT_REPO,
    repo_type='dataset',
)

size_mb = os.path.getsize(FILTERED_PARQUET) / 1024 / 1024
print(f'Uploading abstracts_filtered.parquet ({size_mb:.0f} MB) ...')
api.upload_file(
    path_or_fileobj=FILTERED_PARQUET,
    path_in_repo='abstracts_filtered.parquet',
    repo_id=HF_OUTPUT_REPO,
    repo_type='dataset',
)

chroma_size_mb = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(PERSIST_DIR) for f in files
) / 1024 / 1024
print(f'Uploading chroma_db ({chroma_size_mb:.0f} MB) ...')
api.upload_folder(
    folder_path=PERSIST_DIR,
    path_in_repo='chroma_db',
    repo_id=HF_OUTPUT_REPO,
    repo_type='dataset',
)

print(f'\n✓ Done. https://huggingface.co/datasets/{HF_OUTPUT_REPO}')